In [64]:
import pandas as pd
from datetime import datetime

In [65]:
file_path = "data/Kabupaten Kota (0).xlsx"  # Ganti dengan path file Excel yang sebenarnya
komponens = {'adhb': (0, 17), 'adhk': (24, 41), 'distribusi': (48, 65), 'implisit': (72, 89), 'growth_yony': (96, 113), 'growth_qtoq': (120, 137), 'growth_ctoc': (144, 161), 'growth_implisit_yony': (168, 185), 'growth_implisit_qtoq': (192, 209)}

Data provinsi

In [66]:
sheet_name = 'Prov'
df_prov = {}

In [67]:
df = pd.read_excel(file_path, sheet_name, skiprows=4)
df.fillna(0, inplace=True)

start_col = df.columns.get_loc(datetime.now().year-5)
end_col = df.columns.get_loc(datetime.now().year-1)+1
for komponen, (start_row, end_row) in komponens.items():
    df_prov[f'df_prov_{komponen}'] = df.iloc[start_row:end_row, start_col:end_col].apply(pd.to_numeric, errors='coerce')

Data kabupaten kota

In [68]:
df_kabkot = {}
for kabkot in range(1, 39):
    sheet_name = f'0{kabkot}' if kabkot < 10 else f'{kabkot}'
    df = pd.read_excel(file_path, sheet_name, skiprows=4)
    df.fillna(0, inplace=True)

    start_col = df.columns.get_loc(datetime.now().year-5)
    end_col = df.columns.get_loc(datetime.now().year-1)+1
    for komponen, (start_row, end_row) in komponens.items():
        if df.iloc[start_row:end_row, start_col:end_col].eq(0).all().all():
            continue
        df_kabkot[f'df_kabkot_{sheet_name}_{komponen}'] = df.iloc[start_row:end_row, start_col:end_col].apply(pd.to_numeric, errors='coerce')

Data rekap/total kabupaten kota per komponen

1. ADHB
2. ADHK

In [69]:
df_kabkot_rekap = {}
df_kabkot_rekap_adhb = []
df_kabkot_rekap_adhk = []
for komponen, df in df_kabkot.items():
    if 'adhb' in komponen:
        df_kabkot_rekap_adhb.append(df)
    elif 'adhk' in komponen:
        df_kabkot_rekap_adhk.append(df)

df_kabkot_rekap['df_kabkot_rekap_adhb'] = sum(df_kabkot_rekap_adhb)
df_kabkot_rekap['df_kabkot_rekap_adhk'] = sum(df_kabkot_rekap_adhk)

print(df_kabkot_rekap['df_kabkot_rekap_adhk'])

            2020         Q1.11         Q2.11         Q3.11         Q4.11  \
24  7.434622e+07  1.805109e+07  1.887310e+07  1.907419e+07  1.921849e+07   
25  3.579616e+07  8.641768e+06  9.004882e+06  9.217463e+06  9.098619e+06   
26  3.666429e+06  7.179387e+05  1.145473e+06  9.444484e+05  9.056403e+05   
27  9.556672e+06  2.285240e+06  2.309231e+06  2.310405e+06  2.364314e+06   
28  4.024709e+06  1.023883e+06  1.041166e+06  1.041362e+06  1.122481e+06   
29  1.546313e+07  3.932394e+06  3.907548e+06  4.063197e+06  4.143355e+06   
30  2.682403e+06  6.887513e+05  6.664180e+05  6.708818e+05  7.207062e+05   
31  3.156711e+06  7.611133e+05  7.983767e+05  8.264327e+05  8.633743e+05   
32  2.519573e+06  6.126158e+05  6.117863e+05  6.274907e+05  6.301340e+05   
33  2.251278e+07  3.354543e+06  6.469205e+06  6.085757e+06  7.477390e+06   
34  4.591422e+07  1.117293e+07  1.120762e+07  1.166372e+07  1.212704e+07   
35  3.059535e+07  7.413449e+06  7.298188e+06  7.744872e+06  8.091961e+06   
36  1.531887

3. Distribusi

In [70]:
# Gantilah 0 menjadi 1 untuk menghindari ZeroDivisionError
denominator = df_kabkot_rekap['df_kabkot_rekap_adhb'].iloc[-1].replace(0, 1)

# Lakukan pembagian
df_kabkot_rekap_distribusi = df_kabkot_rekap['df_kabkot_rekap_adhb'].div(denominator) * 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_distribusi'] = df_kabkot_rekap_distribusi

print(df_kabkot_rekap['df_kabkot_rekap_distribusi'])

          2020       Q1.11       Q2.11       Q3.11       Q4.11        2021  \
0    62.515554   60.968592   59.201040   57.943743   55.428540   58.253399   
1    30.218541   29.336285   28.161644   28.354325   26.583726   28.048323   
2     3.422008    2.656650    4.009958    3.137822    2.857004    3.165950   
3     8.023879    7.737814    7.263114    6.969039    6.727657    7.149678   
4     3.429271    3.518093    3.350722    3.213362    3.308127    3.341836   
5    12.250620   12.547643   11.514311   11.509096   11.116122   11.638939   
6     2.413750    2.496432    2.251593    2.171219    2.229255    2.280396   
7     2.757485    2.675675    2.649698    2.588879    2.606648    2.628277   
8     2.192481    2.108901    1.956212    1.927956    1.870759    1.960339   
9    21.093675   12.275110   23.665573   20.919559   24.164583   20.514757   
10   37.289765   36.900138   34.428114   34.587394   34.032346   34.921720   
11   24.443093   23.943308   21.903991   22.526896   22.299999  

4. Implisit

In [71]:
# Gantilah 0 menjadi 1 untuk menghindari ZeroDivisionError
denominator = df_kabkot_rekap['df_kabkot_rekap_adhk'].replace(0, 1)

# Melakukan reindexing denominator sesuai dengan index df_kabkot_rekap['df_kabkot_rekap_adhb']
denominator.index = df_kabkot_rekap['df_kabkot_rekap_adhb'].index

# Lakukan pembagian
df_kabkot_rekap_implisit = df_kabkot_rekap['df_kabkot_rekap_adhb'].div(denominator) * 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_implisit'] = df_kabkot_rekap_implisit

print(df_kabkot_rekap['df_kabkot_rekap_implisit'].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 21 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   2020    17 non-null     float64
 1   Q1.11   17 non-null     float64
 2   Q2.11   17 non-null     float64
 3   Q3.11   17 non-null     float64
 4   Q4.11   17 non-null     float64
 5   2021    17 non-null     float64
 6   Q1.12   17 non-null     float64
 7   Q2.12   17 non-null     float64
 8   Q3.12   17 non-null     float64
 9   Q4.12   17 non-null     float64
 10  2022    17 non-null     float64
 11  Q1.13   17 non-null     float64
 12  Q2.13   17 non-null     float64
 13  Q3.13   17 non-null     float64
 14  Q4.13   17 non-null     float64
 15  2023    17 non-null     float64
 16  Q1.14   17 non-null     float64
 17  Q2.14   17 non-null     float64
 18  Q3.14   17 non-null     float64
 19  Q4.14   17 non-null     float64
 20  2024    17 non-null     float64
dtypes: float64(21)
memory usage: 2.9 KB
None


5. Growth Y-on-Y

In [72]:
# Assign adhk_current_year dan adhk_prev_year
adhk_current_year = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 5:]
adhk_prev_year = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, :df_kabkot_rekap['df_kabkot_rekap_adhk'].shape[1] - 5]

# Menyamakan kolom adhk_current_year dan adhk_prev_year
adhk_prev_year.columns = adhk_current_year.columns

# Lakukan pembagian
df_kabkot_rekap_growth_yony = adhk_current_year.div(adhk_prev_year) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_yony'] = df_kabkot_rekap_growth_yony

print(df_kabkot_rekap['df_kabkot_rekap_growth_yony'])

        2021      Q1.12      Q2.12      Q3.12       Q4.12       2022  \
24  1.171071   5.062713   3.692185   2.005782   -1.236496   2.334124   
25  0.465324   5.435086   2.807607   0.109248   -3.642099   1.115595   
26  1.283857  10.613534   1.306421   3.627180    0.969253   3.613791   
27 -3.008182   0.483633   3.622957   1.518386    0.180971   1.446450   
28  5.073247   7.419006   5.523282  13.243572    3.760865   7.415584   
29  3.772630   4.577902   4.187670   2.809553   -0.025209   2.846538   
30  2.399115   3.643207  11.564178  11.795434    5.451783   8.030675   
31  2.932977   9.967069   5.908634  -3.391161    0.026218   2.930927   
32 -1.490189   3.911266   5.018390   5.544332    6.646680   5.291482   
33  3.882736   0.437213 -16.373111  -5.035187    6.125906  -3.818014   
34  0.559938  -0.180185  -0.494206  -0.043850   -0.802113  -0.385321   
35 -0.153219  -0.263121  -0.669989   1.656191    1.715227   0.650317   
36  1.984278  -0.016640  -0.166051  -3.403658   -5.850402  -2.41

6. Growth Q-to-Q

In [73]:
# Assign adhk_current_quarter
adhk_current_quarter = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 6:]
for i in range(1, 4):
    del adhk_current_quarter[datetime.now().year - i]  
# print(adhk_current_quarter.info())

# Assign adhk_prev_quarter
adhk_prev_quarter = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 4:]
for i in range(1, 5):
    del adhk_prev_quarter[datetime.now().year - i] 
del adhk_prev_quarter[adhk_prev_quarter.columns[-1]]
# print(adhk_prev_quarter.info())

# Menyamakan kolom adhk_current_year dan adhk_prev_year
adhk_prev_quarter.columns = adhk_current_quarter.columns

# Lakukan pembagian
df_kabkot_rekap_growth_qtoq = adhk_current_quarter.div(adhk_prev_quarter) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_qtoq'] = df_kabkot_rekap_growth_qtoq

print(df_kabkot_rekap['df_kabkot_rekap_growth_qtoq'])

         Q1.12      Q2.12      Q3.12      Q4.12      Q1.13      Q2.13  \
24   -1.319182   3.189894  -0.578173  -2.446053   2.949796   2.392419   
25    0.141085   1.605103  -0.325900  -4.988277   6.065318   0.976880   
26  -12.312058  46.125599 -15.660702  -6.568573  -2.908882  19.709850   
27   -2.877045   4.206858  -1.981182   0.985168   1.452376   1.606999   
28   -2.016626  -0.106555   7.336433  -1.236370  -3.540775   2.795183   
29   -0.746725  -1.002628   2.607868  -0.838903   2.062103   2.497954   
30   -0.952163   4.152148   0.878489   1.330954  -1.882590   3.085372   
31   -3.057802   1.024625  -5.575403   8.165459  -1.104420   0.355462   
32    1.022465   0.928601   3.080635   1.470088  -1.913251   1.910669   
33  -54.941370  60.571633   6.826873  37.307445 -55.919917  71.025637   
34   -8.033600  -0.005077   4.540605   3.183505  -5.469350   2.893424   
35   -8.626067  -1.956361   8.605684   4.542211  -6.906969   1.813708   
36   -6.845461   3.833241  -3.009840   0.357623  -2

7. Growth C-to-C

In [ ]:
# Assign adhk_current_cumulative
adhk_current_cumulative = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 6:].astype(float)

# Identifikasi kolom dengan pola 'Q1.x, Q2.x, Q3.x, Q4.x'
pattern_cols = [col for col in df_kabkot_rekap['df_kabkot_rekap_adhk'].columns if isinstance(col, str) and '.' in col]

# Dapatkan semua suffix unik (1, 2, 3, ...)
suffixes = set(col.split('.')[1] for col in pattern_cols)

for index, row in adhk_current_cumulative.iterrows():
    for suffix in suffixes:
        q1_col = f'Q1.{suffix}'
        q2_col = f'Q2.{suffix}'
        q3_col = f'Q3.{suffix}'
        q4_col = f'Q4.{suffix}'
        
        # Pastikan kolom ada di DataFrame sebelum memproses
        if q1_col in adhk_current_cumulative.columns and q2_col in adhk_current_cumulative.columns and q3_col in adhk_current_cumulative.columns and q4_col in adhk_current_cumulative.columns:
            q1 = row[q1_col]
            q2 = row[q2_col]
            q3 = row[q3_col]
            q4 = row[q4_col]

            # Terapkan aturan perubahan
            adhk_current_cumulative.at[index, q1_col] = q1
            adhk_current_cumulative.at[index, q2_col] = q1 + q2
            adhk_current_cumulative.at[index, q3_col] = q1 + q2 + q3
            adhk_current_cumulative.at[index, q4_col] = q1 + q2 + q3 + q4

# print(adhk_current_cumulative.info())

# Assign adhk_prev_cumulative
adhk_prev_cumulative = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 1:df.shape[1] - 5].astype(float)

# Identifikasi kolom dengan pola 'Q1.x, Q2.x, Q3.x, Q4.x'
pattern_cols = [col for col in df_kabkot_rekap['df_kabkot_rekap_adhk'].columns if isinstance(col, str) and '.' in col]

# Dapatkan semua suffix unik (1, 2, 3, ...)
suffixes = set(col.split('.')[1] for col in pattern_cols)

for index, row in adhk_prev_cumulative.iterrows():
    for suffix in suffixes:
        q1_col = f'Q1.{suffix}'
        q2_col = f'Q2.{suffix}'
        q3_col = f'Q3.{suffix}'
        q4_col = f'Q4.{suffix}'
        
        # Pastikan kolom ada di DataFrame sebelum memproses
        if q1_col in adhk_prev_cumulative.columns and q2_col in adhk_prev_cumulative.columns and q3_col in adhk_prev_cumulative.columns and q4_col in adhk_prev_cumulative.columns:
            q1 = row[q1_col]
            q2 = row[q2_col]
            q3 = row[q3_col]
            q4 = row[q4_col]

            # Terapkan aturan perubahan
            adhk_prev_cumulative.at[index, q1_col] = q1
            adhk_prev_cumulative.at[index, q2_col] = q1 + q2
            adhk_prev_cumulative.at[index, q3_col] = q1 + q2 + q3
            adhk_prev_cumulative.at[index, q4_col] = q1 + q2 + q3 + q4

# print(adhk_prev_cumulative.info())

# Menyamakan kolom adhk_current_year dan adhk_prev_year
adhk_prev_cumulative.columns = adhk_current_cumulative.columns

# Lakukan pembagian
df_kabkot_rekap_growth_ctoc = adhk_current_cumulative.div(adhk_prev_cumulative) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_ctoc'] = df_kabkot_rekap_growth_ctoc

print(df_kabkot_rekap['df_kabkot_rekap_growth_ctoc'])

        Q1.12      Q2.12     Q3.12      Q4.12       2022      Q1.13  \
24   5.062713   4.362194  3.559552   2.334124   2.334124   3.036059   
25   5.435086   4.094314  2.726981   1.115595   1.115595   2.058325   
26  10.613534   4.892282  4.466754   3.613791   3.613791  11.796644   
27   0.483633   2.061492  1.879766   1.446450   1.446450   4.646708   
28   7.419006   6.463211  8.736192   7.415584   7.415584   2.146846   
29   4.577902   4.383405  3.846162   2.846538   2.846538   2.804037   
30   3.643207   7.538424  8.948038   8.030675   8.030675   4.461199   
31   9.967069   7.889364  3.982031   2.930927   2.930927   2.041743   
32   3.911266   4.464453  4.830356   5.291482   5.291482   3.547523   
33   0.437213 -10.632842 -8.491608  -3.818014  -3.818014   3.821148   
34  -0.180185  -0.337439 -0.236854  -0.385321  -0.385321   1.963768   
35  -0.263121  -0.464961  0.266589   0.650317   0.650317   3.628886   
36  -0.016640  -0.092806 -1.212500  -2.410379  -2.410379  -1.311640   
37  16

8. Growth implisit Y-on-Y

In [75]:
# Assign implisit_current_year dan implisit_prev_year
implisit_current_year = df_kabkot_rekap['df_kabkot_rekap_implisit'].iloc[:, 5:]
implisit_prev_year = df_kabkot_rekap['df_kabkot_rekap_implisit'].iloc[:, :df_kabkot_rekap['df_kabkot_rekap_implisit'].shape[1] - 5]

# Menyamakan kolom implisit_current_year dan implisit_prev_year
implisit_prev_year.columns = implisit_current_year.columns

# Lakukan pembagian
df_kabkot_rekap_growth_implisit_yony = implisit_current_year.div(implisit_prev_year) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_implisit_yony'] = df_kabkot_rekap_growth_implisit_yony

print(df_kabkot_rekap['df_kabkot_rekap_growth_implisit_yony'])

        2021      Q1.12      Q2.12      Q3.12      Q4.12       2022  \
0   1.870410   2.431199   4.290248   3.708037   5.302192   3.919305   
1   2.185306   2.748480   5.659959   3.635100   5.600311   4.356480   
2   1.030855   2.816332   1.540829   2.201740   2.807709   2.241791   
3   1.610407   2.219850   2.301494   3.045413   5.111227   3.177132   
4   2.579919   2.898807   2.414516   2.217138   2.841916   2.580751   
5   1.261354   0.676935   4.177988   4.892673   6.448891   4.055290   
6   2.045300   2.837438   2.698666   4.196067   5.133210   3.741704   
7   2.417607   5.430886   3.570659   4.226873   4.095685   4.300038   
8   0.389204   3.487815   4.161436   3.595054   1.287815   3.112864   
9   3.548063   7.278506   0.126823   2.929624   5.439432   3.633852   
10  3.003553   3.528379   4.429569   3.143907   3.957685   3.760080   
11  2.574030   4.527133   4.674649   3.098056   3.621670   3.958555   
12  3.735563   1.676651   3.981428   3.430967   4.938642   3.507932   
13  5.

9. Growth implisit Q-to-Q

In [76]:
# Assign implisit_current_quarter
implisit_current_quarter = df_kabkot_rekap['df_kabkot_rekap_implisit'].iloc[:, 6:]
for i in range(1, 4):
    del implisit_current_quarter[datetime.now().year - i]  
# print(implisit_current_quarter.info())

# Assign implisit_prev_quarter
implisit_prev_quarter = df_kabkot_rekap['df_kabkot_rekap_implisit'].iloc[:, 4:]
for i in range(1, 5):
    del implisit_prev_quarter[datetime.now().year - i] 
del implisit_prev_quarter[implisit_prev_quarter.columns[-1]]
# print(implisit_prev_quarter.info())

# Menyamakan kolom implisit_current_year dan implisit_prev_year
implisit_prev_quarter.columns = implisit_current_quarter.columns

# Lakukan pembagian
df_kabkot_rekap_growth_implisit_qtoq = implisit_current_quarter.div(implisit_prev_quarter) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_implisit_qtoq'] = df_kabkot_rekap_growth_implisit_qtoq

print(df_kabkot_rekap['df_kabkot_rekap_growth_implisit_qtoq'])

       Q1.12     Q2.12     Q3.12      Q4.12      Q1.13     Q2.13     Q3.13  \
0   0.006940  2.142543  0.914021   2.152522   0.155896  0.168732  0.355567   
1  -0.471647  2.335020  1.095760   2.556021   0.042161  0.389026  0.527528   
2   0.545634  0.924889  0.097093   1.214496   0.635159  0.487532  0.559100   
3   1.408053  0.421673  1.224251   1.968173   0.376182 -0.436277 -0.179583   
4   0.016906  0.698791  0.279039   1.826854   0.051153  1.064645  0.107297   
5  -0.174107  3.225338  1.418568   1.857767   0.227168 -1.071730  0.769632   
6   0.465356  0.556765  1.837919   2.188680   0.213683  0.438623  0.188673   
7   2.347663  0.180980 -0.466967   2.000497   0.053265  2.307615 -1.681405   
8   0.041811  0.989835  0.141580   0.111404   0.373884  0.694400  1.295217   
9   1.270919  0.791432  1.220923   2.052676  -1.133496  3.729380 -3.725166   
10  1.576296  1.347904 -0.089896   1.074137   0.338931  1.778429  1.488953   
11  2.129472  0.523746  0.022207   0.910046   0.309144  2.504398

In [77]:
# # Pemeriksaan index dan columns
# print(adhk_current_cumulative.index.equals(adhk_prev_cumulative.index))  # Harus True
# print(adhk_current_cumulative.columns.equals(adhk_prev_cumulative.columns))  # Harus True